In [1]:
STUDENT_ID = "TUPM-26-1227"       
SEED_DIGIT = int(STUDENT_ID[-1])

TARGET_OS    = "Android" if SEED_DIGIT % 2 == 0 else "iOS"   
TARGET_CLASS = (SEED_DIGIT % 5) + 1                          

print("SEED_DIGIT      :", SEED_DIGIT)
print("Operating system:", TARGET_OS)
print("Behavior class  :", TARGET_CLASS)

SEED_DIGIT      : 7
Operating system: iOS
Behavior class  : 3


In [ ]:
import os, csv, json, math, shutil, filecmp

def locate_file(filename, keyword):
    target = filename.lower()
    partial = None
    for root, dirs, files in os.walk("."):
        dirs[:] = [d for d in dirs if not d.startswith(".") and d not in ("output", "backup")]
        for f in files:
            if f.lower() == target:
                return os.path.join(root, f)
            if partial is None and keyword in f.lower() and f.lower().endswith(".csv"):
                partial = os.path.join(root, f)
    if partial:
        return partial
    raise FileNotFoundError(
        f"Could not find {filename} under {os.getcwd()}. "
        f"Place it next to this notebook or set DATASET_PATH manually.")

def find_col(header, key, exact=False, required=True):
    for i, name in enumerate(header):
        n = name.strip().lower()
        if (n == key) if exact else (key in n):
            return i
    if required:
        raise ValueError(f"Column containing '{key}' not found in header: {header}")
    return None

def to_float(text):
    try:
        v = float(str(text).strip().replace(",", ""))
    except (ValueError, TypeError):
        return None
    return v if math.isfinite(v) and v >= 0 else None

DATASET_PATH = None            
if DATASET_PATH is None:
    DATASET_PATH = locate_file("user_behavior_dataset.csv", "user_behavior")
print("Dataset:", DATASET_PATH)

for d in ("output", "backup"):
    os.makedirs(d, exist_ok=True)

EXPORT_CSV   = os.path.join("output", f"{TARGET_OS.lower()}_class{TARGET_CLASS}_users.csv")
SUMMARY_JSON = os.path.join("output", "usage_summary.json")
BACKUP_CSV   = os.path.join("backup", f"{TARGET_OS.lower()}_class{TARGET_CLASS}_users_backup.csv")

Dataset: .\data\user_behavior_dataset.csv


In [ ]:
total_records = 0
os_matches = 0
class_matches = 0
skipped_rows = 0
battery_sum = data_sum = 0.0
battery_n = data_n = 0

with open(DATASET_PATH, "r", newline="", encoding="utf-8-sig") as src, \
    open(EXPORT_CSV, "w", newline="", encoding="utf-8") as out:

    reader = csv.reader(src)
    writer = csv.writer(out)
    header = next(reader)

    i_id    = find_col(header, "user id")
    i_model = find_col(header, "device model")
    i_os    = find_col(header, "operating system")
    i_batt  = find_col(header, "battery drain")
    i_data  = find_col(header, "data usage")
    i_class = find_col(header, "behavior class")
    export_idx = [i_id, i_model, i_os, i_batt, i_data]
    max_idx = max(export_idx + [i_class])

    writer.writerow([header[i].strip() for i in export_idx])

    for row in reader:
        if not any(cell.strip() for cell in row):     
            continue
        total_records += 1
        if len(row) <= max_idx:                       
            skipped_rows += 1
            continue
        if row[i_os].strip().lower() != TARGET_OS.lower():
            continue
        os_matches += 1
        try:
            behavior = int(float(row[i_class].strip()))
        except ValueError:
            skipped_rows += 1
            continue
        if behavior != TARGET_CLASS:
            continue

        class_matches += 1
        writer.writerow([row[i].strip() for i in export_idx])

        b = to_float(row[i_batt])
        if b is not None:
            battery_sum += b; battery_n += 1
        d = to_float(row[i_data])
        if d is not None:
            data_sum += d; data_n += 1

avg_battery = round(battery_sum / battery_n, 2) if battery_n else None   
avg_data    = round(data_sum / data_n, 2) if data_n else None
print("Rows read:", total_records, "| malformed/invalid skipped:", skipped_rows)
if class_matches == 0:
    print("No records matched the selected OS and behavior class - averages are N/A.")

Rows read: 700 | malformed/invalid skipped: 0


In [ ]:
with open(EXPORT_CSV, "r", newline="", encoding="utf-8") as f:
    exported_rows = sum(1 for _ in csv.reader(f)) - 1          

count_verified = (exported_rows == class_matches)

summary = {
    "student_id": STUDENT_ID,
    "operating_system": TARGET_OS,
    "user_behavior_class": TARGET_CLASS,
    "total_records_in_dataset": total_records,
    "records_matching_os": os_matches,
    "records_in_target_class": class_matches,
    "average_battery_drain_mAh_per_day": avg_battery,
    "average_data_usage_MB_per_day": avg_data,
    "exported_row_count": exported_rows,
    "count_verified": count_verified,
    "exported_csv_file": EXPORT_CSV,
}
with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)
print("Saved:", SUMMARY_JSON)
print("Exported rows == selected records:", count_verified)

Saved: output\usage_summary.json
Exported rows == selected records: True


In [5]:
backup_ok, backup_size = False, 0
try:
    shutil.copy2(EXPORT_CSV, BACKUP_CSV)
    backup_ok = os.path.isfile(BACKUP_CSV) and filecmp.cmp(EXPORT_CSV, BACKUP_CSV, shallow=False)
    backup_size = os.path.getsize(BACKUP_CSV)
except (FileNotFoundError, PermissionError) as e:
    print("Backup failed:", e)
print("Backup verified:", backup_ok, "| size:", backup_size, "bytes")

Backup verified: True | size: 973 bytes


In [6]:
fmt = lambda v: "N/A (no matching records)" if v is None else f"{v}"

print(f"Operating system and behavior class : {TARGET_OS}, Class {TARGET_CLASS}")
print(f"Total records in dataset            : {total_records}")
print(f"Records matching operating system   : {os_matches}")
print(f"Records in target behavior class    : {class_matches}")
print(f"Average battery drain (mAh/day)     : {fmt(avg_battery)}")
print(f"Average data usage (MB/day)         : {fmt(avg_data)}")
print(f"Count verification                  : {'Matched' if count_verified else 'MISMATCH'} ({exported_rows} exported = {class_matches} selected)")
print(f"Exported CSV file                   : {EXPORT_CSV}")
print(f"JSON summary file                   : {SUMMARY_JSON}")
print(f"Backup file and size                : {BACKUP_CSV} ({backup_size} bytes)")

Operating system and behavior class : iOS, Class 3
Total records in dataset            : 700
Records matching operating system   : 146
Records in target behavior class    : 32
Average battery drain (mAh/day)     : 1492.03
Average data usage (MB/day)         : 828.09
Count verification                  : Matched (32 exported = 32 selected)
Exported CSV file                   : output\ios_class3_users.csv
JSON summary file                   : output\usage_summary.json
Backup file and size                : backup\ios_class3_users_backup.csv (973 bytes)
